In [1]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os 

no null values obtained

In [ ]:


# 1. Path to the folder containing CSV files
from pathlib import Path; FOLDER_PATH = str((Path.cwd() / 'ohlcv_data') if (Path.cwd() / 'ohlcv_data').exists() else (Path.cwd().parent / 'ohlcv_data'))

# 2. Optional: store processed results
null_values = []

# 3. Loop through all files in the folder
for filename in os.listdir(FOLDER_PATH):
    if filename.endswith(".csv"):
        file_path = os.path.join(FOLDER_PATH, filename)
        
        # 4. Read CSV
        df = pd.read_csv(file_path)
        if df.isnull().any().any():
            null_values.append(filename)


        
        
null_values


[]

supporting required primary functions

In [2]:
def calculate_mean_std_z_skew_kurt(df, window):
    roll = df['Returns'].rolling(window=window)

    df[f'Mean_{window}'] = roll.mean()
    df[f'Std_{window}'] = roll.std()
    df[f'Z_Score_{window}'] = (df['Returns'] - df[f'Mean_{window}']) / df[f'Std_{window}']
    df[f'Skew_{window}'] = roll.skew()
    df[f'Kurt_{window}'] = roll.kurt()   

    return (
        df[f'Mean_{window}'],
        df[f'Std_{window}'],
        df[f'Z_Score_{window}'],
        df[f'Skew_{window}'],
        df[f'Kurt_{window}']
    )


def rolling_parkinson_vol(df, window):
    hl_log_sq = np.log(df['High'] / df['Low']) ** 2
    vol = np.sqrt(
        hl_log_sq.rolling(window).mean() / (4 * np.log(2))
    )
    return vol


def rolling_garman_klass_vol(df, window):
    hl = np.log(df['High'] / df['Low']) ** 2
    co = np.log(df['Close'] / df['Open']) ** 2

    vol = np.sqrt(
        (0.5 * hl - (2 * np.log(2) - 1) * co).rolling(window).mean()
    )
    return vol


def rolling_yang_zhang_vol(df, window):
    log_ho = np.log(df['High'] / df['Open'])
    log_lo = np.log(df['Low'] / df['Open'])
    log_co = np.log(df['Close'] / df['Open'])
    log_oc = np.log(df['Open'] / df['Close'].shift(1))

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)

    sigma_oc = log_oc.rolling(window).var()
    sigma_co = log_co.rolling(window).var()
    sigma_rs = rs.rolling(window).mean()

    k = 0.34 / (1.34 + (window + 1) / (window - 1))

    vol = np.sqrt(sigma_oc + k * sigma_co + (1 - k) * sigma_rs)
    return vol



def rolling_linear_fit(series, window):
    slopes = np.full(len(series), np.nan)
    mses = np.full(len(series), np.nan)

    x = np.arange(window)

    for i in range(window - 1, len(series)):
        y = series.iloc[i - window + 1:i + 1].values

        # OLS slope
        x_mean = x.mean()
        y_mean = y.mean()

        num = np.sum((x - x_mean) * (y - y_mean))
        den = np.sum((x - x_mean) ** 2)

        slope = num / den
        intercept = y_mean - slope * x_mean

        y_hat = slope * x + intercept
        mse = np.mean((y - y_hat) ** 2)

        slopes[i] = slope
        mses[i] = mse

    return slopes, mses



In [3]:
from pathlib import Path; FOLDER_PATH = str((Path.cwd() / 'ohlcv_data') if (Path.cwd() / 'ohlcv_data').exists() else (Path.cwd().parent / 'ohlcv_data'))
OUTPUT_FOLDER = str((Path.cwd() / 'ohlcv_data_with_features') if (Path.cwd() / 'ohlcv_data_with_features').exists() else (Path.cwd().parent / 'ohlcv_data_with_features'))

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

for filename in os.listdir(FOLDER_PATH):
    if not filename.endswith(".csv"):
        continue

    file_path = os.path.join(FOLDER_PATH, filename)
    df = pd.read_csv(file_path)

    # ---------- standardized features ----------
    df['OH_ratio'] = (df['Open'] - df['High']) / df['Open']
    df['OL_ratio'] = (df['Open'] - df['Low']) / df['Open']
    df['CH_ratio'] = (df['Close'] - df['High']) / df['Close']
    df['CL_ratio'] = (df['Close'] - df['Low']) / df['Close']
    df['Price_Change'] = (df['Close'] - df['Open']) / df['Open']
    df['Returns'] = df['Close'].pct_change()
    df['Day_Range'] = (df['High'] - df['Low']) / df['Low']

    # ---------- statistical features ----------
    for w in [5, 20, 50, 100, 200]:
        (df[f'Mean_{w}'],
         df[f'Std_{w}'],
         df[f'Z_score_{w}'],
         df[f'Skew_{w}'],
         df[f'Kurt_{w}']) = calculate_mean_std_z_skew_kurt(df, window=w)

    # ---------- volatility features ----------
    for w in [5, 20, 50, 100, 200]:
        df[f'Parkinson_Vol_{w}'] = rolling_parkinson_vol(df, w)
        df[f'GK_Vol_{w}'] = rolling_garman_klass_vol(df, w)
        df[f'YZ_Vol_{w}'] = rolling_yang_zhang_vol(df, w)

    # ---------- regression features ----------
    for w in [5, 20, 50, 100, 200]:
        slope, mse = rolling_linear_fit(df['Returns'], w)
        df[f'Returns_Slope_{w}'] = slope
        df[f'Returns_MSE_{w}'] = mse

    # ---------- save ----------
    output_path = os.path.join(OUTPUT_FOLDER, filename)
    df.to_csv(output_path, index=False)

    print(f"Processed: {filename}")


Processed: AARTIIND.NS.csv
Processed: AARTIPHARM.NS.csv
Processed: AAVAS.NS.csv
Processed: ABDL.NS.csv
Processed: ABFRL.NS.csv
Processed: ABLBL.NS.csv
Processed: ACE.NS.csv
Processed: ACI.NS.csv
Processed: ACMESOLAR.NS.csv
Processed: ACUTAAS.NS.csv
Processed: AEQUS.NS.csv
Processed: AETHER.NS.csv
Processed: AFCONS.NS.csv
Processed: AGI.NS.csv
Processed: AHLUCONT.NS.csv
Processed: AJAXENGG.NS.csv
Processed: AKUMS.NS.csv
Processed: AKZOINDIA.NS.csv
Processed: ALIVUS.NS.csv
Processed: ALKYLAMINE.NS.csv
Processed: ALOKINDS.NS.csv
Processed: ANURAS.NS.csv
Processed: APOLLO.NS.csv
Processed: APTUS.NS.csv
Processed: ARVIND.NS.csv
Processed: ARVINDFASN.NS.csv
Processed: ASHAPURMIN.NS.csv
Processed: ASKAUTOLTD.NS.csv
Processed: ASTRAMICRO.NS.csv
Processed: ATLANTAELE.NS.csv
Processed: AURIONPRO.NS.csv
Processed: AVALON.NS.csv
Processed: AVANTIFEED.NS.csv
Processed: AVL.NS.csv
Processed: AXISCADES.NS.csv
Processed: AZAD.NS.csv
Processed: BAJAJELEC.NS.csv
Processed: BALRAMCHIN.NS.csv
Processed: B